# Clean v6 + high-earner identification — ML1 Task 2 salary prediction

This notebook keeps the strong v6 foundation and the best clean compression blocks found so far, then tests a **narrow high-tail identification strategy**.

**What stays from v6**
- Same train / validation / internal-test split.
- Raw validation and internal-test targets.
- `SVR(kernel="rbf")` with the known strong setting: `C=0.2`, `gamma=0.0007`, `epsilon=0.05`.
- `log_floor500` target transformation as the baseline.
- `high_mild` sample weights.
- Existing v6-style dense target encodings, multi-select salary scores, top technology indicators, missing flags, and experience features.

**What we keep from the clean notebooks**
- `tech_families`, because it improved validation performance.
- `work_profile`, because it added another small validation improvement.

**What this notebook tests for the high-salary problem**
- Upper-distribution encodings: smoothed group `log_salary_q75` and `log_salary_q90` for region/role/work combinations.
- Logistic high-earner probability features for `salary > 100k` and `salary > 150k` are implemented and can be enabled with `RUN_LOGISTIC_PROBABILITY_FEATURES = True`. They are optional because they made RBF-SVR fitting much slower during testing.
- A small target-variant check (`sqrt_floor500`) is implemented and can be enabled with `RUN_SQRT_TARGET_CANDIDATE = True`. It is optional because it slowed down SVR fitting in this dataset.

No PCA, no trees, no boosting, no unbounded tail uplift, and no raw-dollar tail specialist.


In [ ]:
# ===============================================================
# 1. Imports and global configuration
# ===============================================================
from __future__ import annotations

import json
import math
import os
import gc
import pickle
import re
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.exceptions import ConvergenceWarning
from pandas.errors import PerformanceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=PerformanceWarning)

RANDOM_STATE = 42
TRAIN_PATH = Path("train.csv")
TEST_PATH = Path("test.csv")
SAMPLE_SUBMISSION_PATH = Path("sample_submission.csv")
OUTPUT_DIR = Path("clean_v6_high_tail_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "annual.pay.usd"
ID_COL = "id"

# Use FAST_MODE first. After the notebook runs end-to-end, set FAST_MODE=False for a wider search.
FAST_MODE = True

# Keep the first run light. Set True after choosing a candidate if you want
# internal-test diagnostics and a final train+validation refit submission.
RUN_INTERNAL_FINAL = True



# The sqrt target is also implemented, but it can slow down SVR substantially;
# keep False for the first clean run and enable only if you want to test a less-compressive target.
RUN_SQRT_TARGET_CANDIDATE = False

# Logistic high-earner probability features are implemented below, but they can make
# the RBF-SVR optimization much slower on this dataset. Keep False for the clean first run;
# set True only if you want to test them after the upper-encoding candidates.
RUN_LOGISTIC_PROBABILITY_FEATURES = False

# Keep the internal-test set for final locked evaluation only.
VAL_SIZE = 0.15
INTERNAL_TEST_SIZE = 0.15
N_SPLITS_CV = 5

# Salary thresholds used for tail diagnostics and engineered risk features.
LOW_SALARY_FLOOR = 500.0
HIGH_SALARY_100K = 100_000.0
HIGH_SALARY_200K = 200_000.0
HIGH_SALARY_500K = 500_000.0
LOW_SALARY_10K = 10_000.0

# Safety clipping for final predictions only. This prevents numerical accidents, not metric gaming.
PRED_MIN_USD = 1.0
PRED_MAX_USD = 2_000_000.0

print("Configuration loaded — clean v6 + high-tail identification notebook.")
print("FAST_MODE:", FAST_MODE)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


Configuration loaded — clean v6 + high-tail identification notebook.
FAST_MODE: True
OUTPUT_DIR: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\clean_v6_high_tail_outputs


In [ ]:
# ===============================================================
# 2. Load data and split
# ===============================================================
def normalize_text_value(value):
    """Normalize text while preserving missing values."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = value.replace("’", "'").replace("‘", "'").replace("`", "'")
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(normalize_text_value)
    # normalize common "Other" labels
    replacements = {
        "Other:": "Other",
        "Other (please specify):": "Other",
    }
    for col in ["dev.role", "industry", "first.help.source"]:
        if col in df.columns:
            df[col] = df[col].replace(replacements)
    return df


def make_region_strata(region_series: pd.Series, min_count: int = 10) -> pd.Series:
    counts = region_series.value_counts(dropna=False)
    return region_series.where(region_series.map(counts) >= min_count, "__RARE_REGION__")


def summarize_salary_split(name: str, y: pd.Series) -> Dict[str, Any]:
    y = pd.Series(y).astype(float)
    return {
        "split": name,
        "n": int(len(y)),
        "mean": float(y.mean()),
        "median": float(y.median()),
        "min": float(y.min()),
        "max": float(y.max()),
        "lt_500": int((y < LOW_SALARY_FLOOR).sum()),
        "gt_100k": int((y > HIGH_SALARY_100K).sum()),
        "gt_200k": int((y > HIGH_SALARY_200K).sum()),
        "p01": float(y.quantile(0.01)),
        "p05": float(y.quantile(0.05)),
        "p25": float(y.quantile(0.25)),
        "p75": float(y.quantile(0.75)),
        "p95": float(y.quantile(0.95)),
        "p99": float(y.quantile(0.99)),
    }


df_all = normalize_object_columns(pd.read_csv(TRAIN_PATH))
df_kaggle = normalize_object_columns(pd.read_csv(TEST_PATH))

assert TARGET in df_all.columns, f"{TARGET} missing from train.csv"
assert ID_COL in df_kaggle.columns, f"{ID_COL} missing from test.csv"

strata_full = make_region_strata(df_all["region"]) if "region" in df_all.columns else None

df_trainval, df_internal_test = train_test_split(
    df_all,
    test_size=INTERNAL_TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=strata_full,
)

strata_trainval = make_region_strata(df_trainval["region"]) if "region" in df_trainval.columns else None
relative_val_size = VAL_SIZE / (1.0 - INTERNAL_TEST_SIZE)

df_train, df_val = train_test_split(
    df_trainval,
    test_size=relative_val_size,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=strata_trainval,
)

# Reset indices so later masks align cleanly.
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_internal_test = df_internal_test.reset_index(drop=True)
df_kaggle = df_kaggle.reset_index(drop=True)

y_train_raw = df_train[TARGET].astype(float).reset_index(drop=True)
y_val_raw = df_val[TARGET].astype(float).reset_index(drop=True)
y_internal_test_raw = df_internal_test[TARGET].astype(float).reset_index(drop=True)

split_summary = pd.DataFrame([
    summarize_salary_split("train", y_train_raw),
    summarize_salary_split("validation", y_val_raw),
    summarize_salary_split("internal_test", y_internal_test_raw),
])

print("Raw shapes")
print("train:", df_all.shape)
print("kaggle:", df_kaggle.shape)
print("\nSplit sizes")
print("train:", df_train.shape)
print("validation:", df_val.shape)
print("internal_test:", df_internal_test.shape)
print("kaggle:", df_kaggle.shape)
print("\nSalary split summary")
display(split_summary)

split_summary.to_csv(OUTPUT_DIR / "split_salary_summary.csv", index=False)


Raw shapes
train: (2512, 41)
kaggle: (628, 41)

Split sizes
train: (1758, 41)
validation: (377, 41)
internal_test: (377, 41)
kaggle: (628, 41)

Salary split summary


,split,n,mean,median,min,max,lt_500,gt_100k,gt_200k,p01,p05,p25,p75,p95,p99
0,train,1758,51287.046075,41854.5,11.0,4773360.0,57,143,13,76.26,882.55,16073.5,67898.0,117887.45,178901.9
1,validation,377,46288.106101,38113.0,1.0,384552.0,13,36,3,183.80,756.20,15122.0,65008.0,121496.40,169134.0
2,internal_test,377,45797.676393,39989.0,25.0,911275.0,11,23,2,148.76,809.20,18590.0,60052.0,116545.40,156503.0


In [ ]:
# ===============================================================
# 3. Feature engineering configuration
# ===============================================================
MULTI_SEP = ";"

NUMERIC_BASE_COLS = [
    "coding.years.total",
    "coding.years.professional",
    "experience.years",
    "job.satisfaction",
]

# Ordinal maps. These create dense numeric features.
ORDINAL_MAPS = {
    "age.group": {"18-24": 21, "25-34": 29.5, "35-44": 39.5, "45-54": 49.5, "55+": 60},
    "education": {
        "Primary/elementary school": 0,
        "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": 1,
        "Some college/university study without earning a degree": 2,
        "Associate degree (A.A., A.S., etc.)": 3,
        "Something else": 3,
        "Bachelor's degree (B.A., B.S., B.Eng., etc.)": 4,
        "Master's degree (M.A., M.S., M.Eng., MBA, etc.)": 5,
        "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": 6,
    },
    "company.size": {
        "Just me - I am a freelancer, sole proprietor, etc.": 0,
        "2 to 9 employees": 1,
        "10 to 19 employees": 2,
        "20 to 99 employees": 3,
        "100 to 499 employees": 4,
        "500 to 999 employees": 5,
        "1,000 to 4,999 employees": 6,
        "5,000 to 9,999 employees": 7,
        "10,000 or more employees": 8,
    },
    "tech.purchase.influence": {
        "I have little or no influence": 0,
        "I have some influence": 1,
        "I have a great deal of influence": 2,
    },
    "ai.sentiment": {
        "Very unfavorable": 0,
        "Unfavorable": 1,
        "Indifferent": 2,
        "Unsure": 2,
        "Favorable": 3,
        "Very favorable": 4,
    },
    "ai.trust": {
        "Highly distrust": 0,
        "Somewhat distrust": 1,
        "Neither trust nor distrust": 2,
        "Somewhat trust": 3,
        "Highly trust": 4,
    },
    "ai.complex.rating": {
        "Very poor at handling complex tasks": 0,
        "Bad at handling complex tasks": 1,
        "Neither good or bad at handling complex tasks": 2,
        "Good, but not great at handling complex tasks": 3,
        "Very well at handling complex tasks": 4,
    },
    "daily.search.time": {
        "Less than 15 minutes a day": 0,
        "15-30 minutes a day": 1,
        "30-60 minutes a day": 2,
        "60-120 minutes a day": 3,
        "Over 120 minutes a day": 4,
    },
    "daily.answer.time": {
        "Less than 15 minutes a day": 0,
        "15-30 minutes a day": 1,
        "30-60 minutes a day": 2,
        "60-120 minutes a day": 3,
        "Over 120 minutes a day": 4,
    },
}

QUASI_ORDINAL_MAPS = {
    "is.dev.professional": {
        "I am a developer by profession": 1,
        "I am not primarily a developer, but I write code sometimes as part of my work/studies": 0,
    },
    "people.manager": {"People manager": 1, "Individual contributor": 0},
    "uses.ai": {"Yes": 2, "No, but I plan to soon": 1, "No, and I don't plan to": 0},
}

# build.vs.buy is not treated as ordinal here. We target-encode it as a category.
CATEGORICAL_TE_COLS = [
    "region",
    "dev.role",
    "industry",
    "employment.type",
    "work.location",
    "company.size",
    "education",
    "cloud.hosting",
    "people.manager",
    "is.dev.professional",
    "uses.ai",
    "build.vs.buy",
    "first.help.source",
    "ai.job.threat",
]

# Interactions for target encoding. These are dense high-signal features.
INTERACTION_TE_PAIRS = [
    ("region", "dev.role"),
    ("region", "employment.type"),
    ("region", "industry"),
    ("region", "work.location"),
]

MULTI_SELECT_COLS = [
    "prog.languages",
    "databases",
    "cloud.platforms",
    "web.frameworks",
    "other.tech",
    "dev.tools",
    "dev.environments",
    "personal.os",
    "work.os",
    "project.mgmt.tools",
    "comm.tools",
    "ai.search.tools",
    "ai.tools.used",
    "side.coding",
    "how.learned.coding",
]

# Feature profiles control how many sparse technology binaries are added.
# top0_dense has zero technology binaries and relies on counts + salary-score features.
FEATURE_PROFILES = {
    "top0_dense": {"top_n_binary_per_multiselect": 0, "include_basic_ohe": False},
    "top3_dense": {"top_n_binary_per_multiselect": 3, "include_basic_ohe": False},
    "top5_dense": {"top_n_binary_per_multiselect": 5, "include_basic_ohe": False},
    "top5_hybrid_ohe": {"top_n_binary_per_multiselect": 5, "include_basic_ohe": True},
}

# Keep search manageable first.
if FAST_MODE:
    # Small but diverse set: no tech binaries vs current best-style top5 binaries.
    FEATURE_PROFILE_LIST = ["top0_dense", "top5_dense"]
else:
    FEATURE_PROFILE_LIST = list(FEATURE_PROFILES.keys())

print("Feature profiles:", FEATURE_PROFILE_LIST)


Feature profiles: ['top0_dense', 'top5_dense']


In [ ]:
# ===============================================================
# 4. Target transformation helper
# ===============================================================
@dataclass
class TargetTransformer:
    variant: str
    scaler_: StandardScaler = field(default_factory=StandardScaler)
    upper_clip_: Optional[float] = None
    lower_clip_: Optional[float] = None

    def _prepare_raw(self, y_raw: pd.Series, fit: bool = False) -> np.ndarray:
        y = pd.Series(y_raw).astype(float).to_numpy()
        y = np.clip(y, PRED_MIN_USD, None)

        if self.variant.endswith("floor500") or "floor500" in self.variant:
            y = np.clip(y, LOW_SALARY_FLOOR, None)

        if "upper_clip" in self.variant:
            if fit:
                self.upper_clip_ = float(np.quantile(y, 0.995))
            y = np.clip(y, None, self.upper_clip_)

        return y

    def _transform_unscaled(self, y_raw: pd.Series, fit: bool = False) -> np.ndarray:
        y = self._prepare_raw(y_raw, fit=fit)
        if self.variant.startswith("log"):
            return np.log(y)
        if self.variant.startswith("sqrt"):
            return np.sqrt(y)
        if self.variant.startswith("cbrt"):
            return np.cbrt(y)
        if self.variant.startswith("raw"):
            return y
        raise ValueError(f"Unknown target variant: {self.variant}")

    def fit(self, y_raw: pd.Series) -> "TargetTransformer":
        z = self._transform_unscaled(y_raw, fit=True).reshape(-1, 1)
        self.scaler_.fit(z)
        return self

    def transform(self, y_raw: pd.Series) -> np.ndarray:
        z = self._transform_unscaled(y_raw, fit=False).reshape(-1, 1)
        return self.scaler_.transform(z).ravel()

    def inverse_transform_to_usd(self, y_pred_scaled: np.ndarray) -> np.ndarray:
        z = self.scaler_.inverse_transform(np.asarray(y_pred_scaled).reshape(-1, 1)).ravel()
        if self.variant.startswith("log"):
            y = np.exp(z)
        elif self.variant.startswith("sqrt"):
            y = np.square(np.clip(z, 0, None))
        elif self.variant.startswith("cbrt"):
            y = np.power(z, 3)
        elif self.variant.startswith("raw"):
            y = z
        else:
            raise ValueError(f"Unknown target variant: {self.variant}")
        return np.clip(y, PRED_MIN_USD, PRED_MAX_USD)

    def inverse_transform_to_log_scale(self, y_pred_scaled: np.ndarray) -> np.ndarray:
        """Return predicted log salary. Useful for smearing calibration on log variants."""
        y_usd = self.inverse_transform_to_usd(y_pred_scaled)
        return np.log(np.clip(y_usd, PRED_MIN_USD, None))


TARGET_VARIANTS = [
    "log_raw",
    "log_floor500",
    "sqrt_raw",
    "sqrt_floor500",
    "cbrt_raw",
    "cbrt_floor500",
    "raw_scaled",
    "raw_floor500_scaled",
    "raw_upper_clip_scaled",
]
if FAST_MODE:
    # The previous run suggested sqrt/raw targets are strongest; cbrt is added to reduce log compression while remaining less extreme than raw.
    TARGET_VARIANTS = ["sqrt_raw", "cbrt_raw", "raw_scaled", "log_floor500"]

print("Target variants:", TARGET_VARIANTS)


Target variants: ['sqrt_raw', 'cbrt_raw', 'raw_scaled', 'log_floor500']


In [ ]:
# ===============================================================
# 5. Dense SVR preprocessor
# ===============================================================
def parse_multiselect_cell(value, sep: str = MULTI_SEP) -> List[str]:
    if pd.isna(value):
        return []
    value = normalize_text_value(value)
    if value == "":
        return []
    return [normalize_text_value(item) for item in str(value).split(sep) if normalize_text_value(item) != ""]


def sanitize_feature_name(name: str) -> str:
    name = normalize_text_value(name)
    name = str(name).lower()
    replacements = {
        " ": "_", "/": "_", ".": "_", "(": "", ")": "", ",": "",
        "'": "", "-": "_", "+": "plus", "#": "sharp", ":": "", "!": "",
        "&": "and", "[": "", "]": "",
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    name = re.sub(r"[^a-z0-9_]+", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name or "unknown"


def exp_bucket_from_prof_years(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    return pd.cut(
        x,
        bins=[-np.inf, 1, 3, 5, 10, 20, np.inf],
        labels=["0_1", "1_3", "3_5", "5_10", "10_20", "20_plus"],
    ).astype(str).replace("nan", "__Missing__")


@dataclass
class SmoothedStats:
    mean_map: Dict[str, float]
    high100_map: Dict[str, float]
    high200_map: Dict[str, float]
    low500_map: Dict[str, float]
    count_map: Dict[str, int]
    global_mean: float
    global_high100: float
    global_high200: float
    global_low500: float


@dataclass
class TailFocusedSVRPreprocessor:
    feature_profile: str = "top5_dense"
    te_smoothing: float = 30.0
    rate_smoothing: float = 50.0
    item_smoothing: float = 20.0
    min_missing_rate: float = 0.10

    numeric_medians_: Dict[str, float] = field(default_factory=dict)
    ordinal_fallbacks_: Dict[str, float] = field(default_factory=dict)
    missing_indicator_cols_: List[str] = field(default_factory=list)
    cat_stats_: Dict[str, SmoothedStats] = field(default_factory=dict)
    interaction_stats_: Dict[str, SmoothedStats] = field(default_factory=dict)
    multi_item_stats_: Dict[str, SmoothedStats] = field(default_factory=dict)
    multi_top_items_: Dict[str, List[str]] = field(default_factory=dict)
    feature_cols_: List[str] = field(default_factory=list)
    zero_variance_cols_: List[str] = field(default_factory=list)
    global_log_mean_: float = 0.0
    global_high100_: float = 0.0
    global_high200_: float = 0.0
    global_low500_: float = 0.0

    def fit(self, X_raw: pd.DataFrame, y_raw: pd.Series) -> "TailFocusedSVRPreprocessor":
        X = normalize_object_columns(X_raw.copy())
        X = X.drop(columns=[TARGET], errors="ignore")
        y = pd.Series(y_raw).astype(float).reset_index(drop=True)
        X = X.reset_index(drop=True)

        y_log = np.log(np.clip(y, PRED_MIN_USD, None))
        self.global_log_mean_ = float(y_log.mean())
        self.global_high100_ = float((y > HIGH_SALARY_100K).mean())
        self.global_high200_ = float((y > HIGH_SALARY_200K).mean())
        self.global_low500_ = float((y < LOW_SALARY_FLOOR).mean())

        missing_rates = X.isna().mean()
        self.missing_indicator_cols_ = [c for c, r in missing_rates.items() if r >= self.min_missing_rate and r > 0]

        # Numeric and ordinal medians.
        X_num = self._make_numeric_features(X, fit_mode=True)
        self.numeric_medians_ = {
            col: float(X_num[col].median()) if X_num[col].notna().any() else 0.0
            for col in X_num.columns
        }
        for col, mapping in {**ORDINAL_MAPS, **QUASI_ORDINAL_MAPS}.items():
            if col in X.columns:
                mapped = X[col].replace({"I don't know": np.nan}).map(mapping)
                self.ordinal_fallbacks_[col] = float(mapped.median()) if mapped.notna().any() else 0.0

        # Target encoding stats for categorical variables.
        self.cat_stats_ = {}
        for col in CATEGORICAL_TE_COLS:
            if col in X.columns:
                values = self._clean_category_series(X[col])
                self.cat_stats_[col] = self._fit_smoothed_stats(values, y)

        # Add experience bucket as a target-encoded pseudo-category.
        if "coding.years.professional" in X.columns:
            exp_bucket = exp_bucket_from_prof_years(X["coding.years.professional"])
            self.cat_stats_["professional_years_bucket"] = self._fit_smoothed_stats(exp_bucket, y)

        # Interaction target encodings.
        self.interaction_stats_ = {}
        for col_a, col_b in INTERACTION_TE_PAIRS:
            if col_a in X.columns and col_b in X.columns:
                key = f"{col_a}__x__{col_b}"
                values = self._interaction_series(X[col_a], X[col_b])
                self.interaction_stats_[key] = self._fit_smoothed_stats(values, y)
        if "region" in X.columns and "coding.years.professional" in X.columns:
            key = "region__x__professional_years_bucket"
            values = self._interaction_series(X["region"], exp_bucket_from_prof_years(X["coding.years.professional"]))
            self.interaction_stats_[key] = self._fit_smoothed_stats(values, y)

        # Multi-select item stats and top items.
        self.multi_item_stats_ = {}
        self.multi_top_items_ = {}
        for col in MULTI_SELECT_COLS:
            if col not in X.columns:
                continue
            parsed = [parse_multiselect_cell(v) for v in X[col]]
            item_rows = []
            for i, items in enumerate(parsed):
                for item in items:
                    item_rows.append((item, y.iloc[i]))
            if item_rows:
                item_df = pd.DataFrame(item_rows, columns=["item", "salary"])
                self.multi_item_stats_[col] = self._fit_smoothed_stats(item_df["item"], item_df["salary"])
                counts = Counter([item for items in parsed for item in items])
                self.multi_top_items_[col] = [item for item, _ in counts.most_common(10)]
            else:
                self.multi_item_stats_[col] = self._empty_stats()
                self.multi_top_items_[col] = []

        # Determine final columns on training data.
        X_tmp = self.transform(X_raw, fit_call=True)
        self.zero_variance_cols_ = X_tmp.columns[X_tmp.nunique(dropna=False) <= 1].tolist()
        X_tmp = X_tmp.drop(columns=self.zero_variance_cols_, errors="ignore")
        self.feature_cols_ = X_tmp.columns.tolist()
        return self

    def transform(self, X_raw: pd.DataFrame, fit_call: bool = False) -> pd.DataFrame:
        X = normalize_object_columns(X_raw.copy())
        X = X.drop(columns=[TARGET], errors="ignore")
        X = X.reset_index(drop=True)
        pieces = []

        # Missing flags.
        miss = pd.DataFrame(index=X.index)
        for col in self.missing_indicator_cols_:
            if col in X.columns:
                miss[f"{col}_missing"] = X[col].isna().astype(float)
        if "company.size" in X.columns:
            miss["company_size_dontknow"] = (X["company.size"] == "I don't know").astype(float)
        if not miss.empty:
            pieces.append(miss)

        # Dense numeric features.
        X_num = self._make_numeric_features(X, fit_mode=False)
        for col, med in self.numeric_medians_.items():
            if col not in X_num.columns:
                X_num[col] = med
            else:
                X_num[col] = X_num[col].fillna(med)
        pieces.append(X_num.astype(float))

        # Target encodings for categories.
        te_df = pd.DataFrame(index=X.index)
        for col, stats in self.cat_stats_.items():
            if col == "professional_years_bucket":
                if "coding.years.professional" in X.columns:
                    values = exp_bucket_from_prof_years(X["coding.years.professional"])
                else:
                    values = pd.Series("__Missing__", index=X.index)
            else:
                values = self._clean_category_series(X[col]) if col in X.columns else pd.Series("__Missing__", index=X.index)
            safe_col = sanitize_feature_name(col)
            self._apply_stats_to_frame(te_df, safe_col, values, stats)

        # Target encodings for interactions.
        for key, stats in self.interaction_stats_.items():
            if key == "region__x__professional_years_bucket":
                if "region" in X.columns and "coding.years.professional" in X.columns:
                    values = self._interaction_series(X["region"], exp_bucket_from_prof_years(X["coding.years.professional"]))
                else:
                    values = pd.Series("__Missing__", index=X.index)
            else:
                parts = key.split("__x__")
                col_a, col_b = parts[0], parts[1]
                if col_a in X.columns and col_b in X.columns:
                    values = self._interaction_series(X[col_a], X[col_b])
                else:
                    values = pd.Series("__Missing__", index=X.index)
            safe_key = sanitize_feature_name(key)
            self._apply_stats_to_frame(te_df, safe_key, values, stats)

        if not te_df.empty:
            pieces.append(te_df.astype(float))

        # Multi-select salary score features and optional top-N binaries.
        multi_df = pd.DataFrame(index=X.index)
        top_n = int(FEATURE_PROFILES[self.feature_profile]["top_n_binary_per_multiselect"])
        for col in MULTI_SELECT_COLS:
            if col not in X.columns or col not in self.multi_item_stats_:
                continue
            parsed = [parse_multiselect_cell(v) for v in X[col]]
            prefix = sanitize_feature_name(col)
            stats = self.multi_item_stats_[col]
            counts = np.array([len(items) for items in parsed], dtype=float)
            multi_df[f"{prefix}_count"] = counts

            # Salary score summaries.
            mean_scores, max_scores, high100_mean, high100_max, high200_mean, low500_mean, low500_max = [], [], [], [], [], [], []
            for items in parsed:
                if len(items) == 0:
                    item_mean = [stats.global_mean]
                    item_high100 = [stats.global_high100]
                    item_high200 = [stats.global_high200]
                    item_low500 = [stats.global_low500]
                else:
                    item_mean = [stats.mean_map.get(item, stats.global_mean) for item in items]
                    item_high100 = [stats.high100_map.get(item, stats.global_high100) for item in items]
                    item_high200 = [stats.high200_map.get(item, stats.global_high200) for item in items]
                    item_low500 = [stats.low500_map.get(item, stats.global_low500) for item in items]
                mean_scores.append(float(np.mean(item_mean)))
                max_scores.append(float(np.max(item_mean)))
                high100_mean.append(float(np.mean(item_high100)))
                high100_max.append(float(np.max(item_high100)))
                high200_mean.append(float(np.mean(item_high200)))
                low500_mean.append(float(np.mean(item_low500)))
                low500_max.append(float(np.max(item_low500)))
            multi_df[f"{prefix}_salary_score_mean"] = mean_scores
            multi_df[f"{prefix}_salary_score_max"] = max_scores
            multi_df[f"{prefix}_high100_score_mean"] = high100_mean
            multi_df[f"{prefix}_high100_score_max"] = high100_max
            multi_df[f"{prefix}_high200_score_mean"] = high200_mean
            multi_df[f"{prefix}_low500_score_mean"] = low500_mean
            multi_df[f"{prefix}_low500_score_max"] = low500_max

            # Optional sparse top-N binaries. Keep small N only.
            if top_n > 0:
                for item in self.multi_top_items_.get(col, [])[:top_n]:
                    item_col = f"{prefix}__{sanitize_feature_name(item)}"
                    multi_df[item_col] = [float(item in items) for items in parsed]

        if not multi_df.empty:
            pieces.append(multi_df.astype(float))

        # Optional basic one-hot for very small categories only.
        if FEATURE_PROFILES[self.feature_profile].get("include_basic_ohe", False):
            small_ohe_cols = ["employment.type", "work.location", "people.manager"]
            ohe_src = pd.DataFrame(index=X.index)
            for col in small_ohe_cols:
                if col in X.columns:
                    ohe_src[col] = X[col].fillna("__Missing__").astype(str)
            if not ohe_src.empty:
                pieces.append(pd.get_dummies(ohe_src, prefix_sep="__", drop_first=True, dtype=float))

        X_out = pd.concat(pieces, axis=1) if pieces else pd.DataFrame(index=X.index)
        X_out = X_out.loc[:, ~X_out.columns.duplicated()].replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)

        if fit_call:
            return X_out

        X_out = X_out.drop(columns=self.zero_variance_cols_, errors="ignore")
        # Align to training columns.
        for col in self.feature_cols_:
            if col not in X_out.columns:
                X_out[col] = 0.0
        extra_cols = [c for c in X_out.columns if c not in self.feature_cols_]
        if extra_cols:
            X_out = X_out.drop(columns=extra_cols)
        return X_out[self.feature_cols_].astype(float)

    def _make_numeric_features(self, X: pd.DataFrame, fit_mode: bool = False) -> pd.DataFrame:
        out = pd.DataFrame(index=X.index)
        # Basic numeric.
        for col in NUMERIC_BASE_COLS:
            if col in X.columns:
                out[sanitize_feature_name(col)] = pd.to_numeric(X[col], errors="coerce")
            else:
                out[sanitize_feature_name(col)] = np.nan

        total = pd.to_numeric(X.get("coding.years.total", pd.Series(np.nan, index=X.index)), errors="coerce")
        prof = pd.to_numeric(X.get("coding.years.professional", pd.Series(np.nan, index=X.index)), errors="coerce")
        exp = pd.to_numeric(X.get("experience.years", pd.Series(np.nan, index=X.index)), errors="coerce")

        out["years_before_professional"] = (total - prof).clip(lower=0)

        # Ordinal / quasi-ordinal.
        for col, mapping in {**ORDINAL_MAPS, **QUASI_ORDINAL_MAPS}.items():
            safe = sanitize_feature_name(col)
            if col in X.columns:
                mapped = X[col].replace({"I don't know": np.nan}).map(mapping)
                fallback = self.ordinal_fallbacks_.get(col, np.nan)
                out[safe] = mapped.fillna(fallback)
            else:
                out[safe] = self.ordinal_fallbacks_.get(col, 0.0)

        # Multi-select counts.
        count_cols = {}
        for col in MULTI_SELECT_COLS:
            if col in X.columns:
                counts = pd.Series([len(parse_multiselect_cell(v)) for v in X[col]], index=X.index).astype(float)
            else:
                counts = pd.Series(0.0, index=X.index)
            c_name = f"{sanitize_feature_name(col)}_raw_count"
            count_cols[col] = counts
            out[c_name] = counts
        total_tech_count = sum(count_cols.values()) if count_cols else pd.Series(0.0, index=X.index)
        out["total_multiselect_count"] = total_tech_count

        # Ratios and dense interactions. Add 1 to denominators for stability.
        out["experience_to_professional_ratio"] = (exp / (prof + 1)).clip(lower=0, upper=10)
        age_mid = out.get("age_group", pd.Series(np.nan, index=X.index))
        out["professional_years_to_age_ratio"] = (prof / age_mid.replace(0, np.nan)).clip(lower=0, upper=1)
        out["years_before_professional_ratio"] = (out["years_before_professional"] / (total + 1)).clip(lower=0, upper=1)

        lang_count = count_cols.get("prog.languages", pd.Series(0.0, index=X.index))
        cloud_count = count_cols.get("cloud.platforms", pd.Series(0.0, index=X.index))
        db_count = count_cols.get("databases", pd.Series(0.0, index=X.index))
        ai_count = count_cols.get("ai.tools.used", pd.Series(0.0, index=X.index))
        out["language_count_x_professional_years"] = lang_count * prof.fillna(0)
        out["cloud_count_x_professional_years"] = cloud_count * prof.fillna(0)
        out["database_count_x_professional_years"] = db_count * prof.fillna(0)
        out["ai_count_x_professional_years"] = ai_count * prof.fillna(0)
        out["total_tech_count_x_professional_years"] = total_tech_count * prof.fillna(0)

        manager = out.get("people_manager", pd.Series(0.0, index=X.index)).fillna(0)
        company = out.get("company_size", pd.Series(0.0, index=X.index)).fillna(0)
        influence = out.get("tech_purchase_influence", pd.Series(0.0, index=X.index)).fillna(0)
        out["manager_x_company_size"] = manager * company
        out["influence_x_company_size"] = influence * company
        out["large_company_flag"] = (company >= 6).astype(float)
        out["senior_professional_10y"] = (prof >= 10).astype(float)
        out["senior_professional_20y"] = (prof >= 20).astype(float)
        out["high_experience_10y"] = (exp >= 10).astype(float)
        out["high_experience_20y"] = (exp >= 20).astype(float)

        if "employment.type" in X.columns:
            emp = X["employment.type"].fillna("__Missing__").astype(str)
            out["is_full_time"] = (emp == "Full-time").astype(float)
            out["is_freelance"] = emp.str.contains("Freelance", case=False, na=False).astype(float)
            out["is_student"] = emp.str.contains("Student", case=False, na=False).astype(float)
            out["is_part_time"] = emp.str.contains("Part-time", case=False, na=False).astype(float)
        if "work.location" in X.columns:
            wl = X["work.location"].fillna("__Missing__").astype(str)
            out["is_remote"] = (wl == "Remote").astype(float)
            out["is_hybrid"] = wl.str.contains("Hybrid", case=False, na=False).astype(float)

        # Tail-oriented dense interactions. These are still derived only from available predictors.
        out["senior_x_large_company"] = out["senior_professional_10y"] * out["large_company_flag"]
        out["senior20_x_large_company"] = out["senior_professional_20y"] * out["large_company_flag"]
        out["manager_x_senior"] = manager * out["senior_professional_10y"]
        out["manager_x_senior20"] = manager * out["senior_professional_20y"]
        out["remote_x_senior"] = out.get("is_remote", pd.Series(0.0, index=X.index)) * out["senior_professional_10y"]
        out["fulltime_x_senior"] = out.get("is_full_time", pd.Series(0.0, index=X.index)) * out["senior_professional_10y"]
        out["freelance_x_low_experience"] = out.get("is_freelance", pd.Series(0.0, index=X.index)) * (prof.fillna(0) <= 1).astype(float)
        out["student_or_parttime_low_salary_risk"] = (out.get("is_student", pd.Series(0.0, index=X.index)) + out.get("is_part_time", pd.Series(0.0, index=X.index))).clip(upper=1)

        return out

    def _clean_category_series(self, s: pd.Series) -> pd.Series:
        return s.fillna("__Missing__").astype(str).map(normalize_text_value).fillna("__Missing__")

    def _interaction_series(self, a: pd.Series, b: pd.Series) -> pd.Series:
        a = self._clean_category_series(a)
        b = self._clean_category_series(b)
        return a.astype(str) + "__x__" + b.astype(str)

    def _fit_smoothed_stats(self, values: pd.Series, y_raw: pd.Series) -> SmoothedStats:
        values = pd.Series(values).fillna("__Missing__").astype(str).reset_index(drop=True)
        y = pd.Series(y_raw).astype(float).reset_index(drop=True)
        y_log = np.log(np.clip(y, PRED_MIN_USD, None))
        df = pd.DataFrame({"value": values, "y": y, "log_y": y_log})
        grouped = df.groupby("value")
        counts = grouped.size()
        mean_log = grouped["log_y"].mean()
        high100 = grouped["y"].apply(lambda x: float((x > HIGH_SALARY_100K).mean()))
        high200 = grouped["y"].apply(lambda x: float((x > HIGH_SALARY_200K).mean()))
        low500 = grouped["y"].apply(lambda x: float((x < LOW_SALARY_FLOOR).mean()))

        global_mean = float(y_log.mean())
        global_high100 = float((y > HIGH_SALARY_100K).mean())
        global_high200 = float((y > HIGH_SALARY_200K).mean())
        global_low500 = float((y < LOW_SALARY_FLOOR).mean())

        mean_smooth = (mean_log * counts + global_mean * self.te_smoothing) / (counts + self.te_smoothing)
        high100_smooth = (high100 * counts + global_high100 * self.rate_smoothing) / (counts + self.rate_smoothing)
        high200_smooth = (high200 * counts + global_high200 * self.rate_smoothing) / (counts + self.rate_smoothing)
        low500_smooth = (low500 * counts + global_low500 * self.rate_smoothing) / (counts + self.rate_smoothing)

        return SmoothedStats(
            mean_map=mean_smooth.to_dict(),
            high100_map=high100_smooth.to_dict(),
            high200_map=high200_smooth.to_dict(),
            low500_map=low500_smooth.to_dict(),
            count_map=counts.astype(int).to_dict(),
            global_mean=global_mean,
            global_high100=global_high100,
            global_high200=global_high200,
            global_low500=global_low500,
        )

    def _empty_stats(self) -> SmoothedStats:
        return SmoothedStats({}, {}, {}, {}, {}, self.global_log_mean_, self.global_high100_, self.global_high200_, self.global_low500_)

    def _apply_stats_to_frame(self, out: pd.DataFrame, prefix: str, values: pd.Series, stats: SmoothedStats) -> None:
        values = pd.Series(values).fillna("__Missing__").astype(str)
        out[f"{prefix}_te_log_mean"] = values.map(stats.mean_map).fillna(stats.global_mean).astype(float)
        out[f"{prefix}_te_high100_rate"] = values.map(stats.high100_map).fillna(stats.global_high100).astype(float)
        out[f"{prefix}_te_high200_rate"] = values.map(stats.high200_map).fillna(stats.global_high200).astype(float)
        out[f"{prefix}_te_low500_rate"] = values.map(stats.low500_map).fillna(stats.global_low500).astype(float)
        out[f"{prefix}_te_log_count"] = np.log1p(values.map(stats.count_map).fillna(0).astype(float))

print("TailFocusedSVRPreprocessor ready.")


TailFocusedSVRPreprocessor ready.


In [ ]:
# ===============================================================
# 6. Compression feature blocks
# ===============================================================
# These blocks are intentionally small and interpretable.
# They build on the v6 feature matrix instead of replacing it.
# No PCA, no trees, no boosting, no outside-ML1 algorithms.

MISSINGNESS_GROUPS = {
    "work": [
        "employment.type", "work.location", "dev.role", "company.size", "people.manager",
        "industry", "tech.purchase.influence", "cloud.hosting", "job.satisfaction"
    ],
    "tech": [
        "prog.languages", "databases", "cloud.platforms", "web.frameworks",
        "other.tech", "dev.tools", "dev.environments"
    ],
    "ai": [
        "ai.search.tools", "ai.tools.used", "uses.ai", "ai.sentiment", "ai.trust",
        "ai.complex.rating", "ai.job.threat", "daily.search.time", "daily.answer.time"
    ],
    "profile": [
        "region", "age.group", "education", "is.dev.professional",
        "coding.years.total", "coding.years.professional", "experience.years"
    ],
}

# This block is carried over from the previous clean notebook because it was the only block
# that improved validation RMSE in that run.
TECH_FAMILIES = {
    "data_ml": [
        "Python", "R", "Scala", "SQL", "Snowflake", "BigQuery", "Databricks SQL",
        "Apache Spark", "Hadoop", "Pandas", "NumPy", "Scikit-Learn", "TensorFlow",
        "Torch/PyTorch", "Keras", "MLflow"
    ],
    "cloud_backend": [
        "Amazon Web Services (AWS)", "Microsoft Azure", "Google Cloud", "Cloudflare",
        "OpenShift", "Kubernetes", "Docker", "Terraform", "Ansible", "Java", "Go",
        "Rust", "Kotlin", "PostgreSQL", "Redis", "RabbitMQ", "Apache Kafka", "Kafka"
    ],
    "mobile": [
        "Swift", "Objective-C", "SwiftUI", "Kotlin", "Dart", "Flutter", "React Native",
        "Android Studio"
    ],
    "microsoft_enterprise": [
        "C#", ".NET Framework", ".NET", "ASP.NET", "ASP.NET Core", "Microsoft Azure",
        "Microsoft SQL Server", "PowerShell", "Visual Studio", "Visual Studio Code"
    ],
    "frontend": [
        "HTML/CSS", "JavaScript", "TypeScript", "React", "Angular", "AngularJS",
        "Vue.js", "Next.js", "Node.js", "jQuery", "Vite", "Webpack", "Yarn", "npm"
    ],
    "devops": [
        "Docker", "Kubernetes", "Ansible", "Terraform", "OpenShift", "Puppet",
        "Chef", "Make", "CMake", "Gradle", "Maven", "Git", "Jira"
    ],
    "rare_functional": [
        "Elixir", "Erlang", "F#", "Haskell", "Clojure", "Lisp", "OCaml", "Scala"
    ],
}

SIDE_LEARNING_FAMILIES = {
    "side_income_business": [
        "Freelance/contract work", "Bootstrapping a business"
    ],
    "side_open_source": [
        "Contribute to open-source projects"
    ],
    "side_hobby_learning": [
        "Hobby", "Professional development or self-paced learning from online courses", "School or academic work"
    ],
    "side_no_external_coding": [
        "I don’t code outside of work", "I don't code outside of work"
    ],
    "learned_formal": [
        "School (i.e., University, College, etc)", "Coding Bootcamp"
    ],
    "learned_online": [
        "Other online resources (e.g., videos, blogs, forum, online community)",
        "Online Courses or Certification"
    ],
    "learned_workplace_social": [
        "On the job training", "Colleague", "Friend or family member"
    ],
    "learned_books_media": [
        "Books / Physical media"
    ],
}

AI_TOOL_FAMILIES = {
    "ai_general_chat": [
        "ChatGPT", "Claude", "Google Gemini", "Bing AI", "Meta AI"
    ],
    "ai_coding_assistant": [
        "GitHub Copilot", "Visual Studio Intellicode", "Tabnine", "Codeium", "Amazon Q", "OpenAI Codex"
    ],
    "ai_answer_search": [
        "Search for answers", "Phind", "Perplexity AI", "You.com", "WolframAlpha"
    ],
    "ai_development_tasks": [
        "Writing code", "Debugging and getting help", "Testing code", "Documenting code",
        "Learning about a codebase", "Committing and reviewing code", "Deployment and monitoring"
    ],
    "ai_business_data_tasks": [
        "Generating content or synthetic data", "Project planning", "Predictive analytics"
    ],
}

OS_TOOLING_FAMILIES = {
    "linux_unix_ecosystem": [
        "Ubuntu", "Debian", "Other Linux-based", "Arch", "Fedora", "Red Hat", "BSD",
        "Windows Subsystem for Linux (WSL)", "Cygwin"
    ],
    "apple_ecosystem": [
        "MacOS", "iOS", "iPadOS", "Xcode"
    ],
    "microsoft_ecosystem": [
        "Windows", "Windows Subsystem for Linux (WSL)", "Visual Studio", "Visual Studio Code",
        "Visual Studio Solution", "MSBuild", "NuGet", "Microsoft Teams", "Azure Devops",
        "Microsoft Planner"
    ],
    "professional_ide": [
        "Visual Studio Code", "IntelliJ IDEA", "Visual Studio", "PyCharm", "Android Studio",
        "WebStorm", "PhpStorm", "Jupyter Notebook/JupyterLab", "Xcode", "Rider", "DataGrip",
        "Eclipse", "Goland", "CLion", "RStudio", "Vim", "Neovim"
    ],
    "collaboration_enterprise": [
        "Jira", "Confluence", "Slack", "Microsoft Teams", "Google Meet", "Zoom", "Miro",
        "Notion", "Azure Devops", "Trello", "Asana", "Clickup", "Linear", "Monday.com"
    ],
    "build_package_tools": [
        "npm", "Pip", "Maven (build tool)", "Gradle", "Yarn", "Webpack", "Vite", "NuGet",
        "MSBuild", "Composer", "Make", "CMake", "Homebrew", "APT", "pnpm", "Bun"
    ],
}

TECH_FAMILY_COLS = MULTI_SELECT_COLS
SIDE_LEARNING_COLS = ["side.coding", "how.learned.coding"]
AI_TOOL_COLS = ["ai.search.tools", "ai.tools.used"]
OS_TOOLING_COLS = [
    "personal.os", "work.os", "dev.environments", "dev.tools", "project.mgmt.tools", "comm.tools"
]


def _safe_numeric(series: pd.Series, fallback: float = 0.0) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(fallback).astype(float)


def _col(df: pd.DataFrame, name: str, default: float = 0.0) -> pd.Series:
    if name in df.columns:
        return pd.to_numeric(df[name], errors="coerce").fillna(default).astype(float)
    return pd.Series(default, index=df.index, dtype=float)


def _missing_count(raw_df: pd.DataFrame, cols: List[str]) -> pd.Series:
    existing = [c for c in cols if c in raw_df.columns]
    if not existing:
        return pd.Series(0, index=raw_df.index, dtype=float)
    return raw_df[existing].isna().sum(axis=1).astype(float)


def _parse_sanitized_item_set(value) -> set:
    """Parse one multi-select cell once and return a sanitized set of selected items."""
    return {sanitize_feature_name(x) for x in parse_multiselect_cell(value)}


def _add_family_features(
    X: pd.DataFrame,
    raw: pd.DataFrame,
    families: Dict[str, List[str]],
    cols: List[str],
    prefix: str,
) -> pd.DataFrame:
    """Add compact count/flag features for semantic families.

    This is intentionally implemented with column-level parsing instead of row-wise
    raw.apply, because this notebook repeats the operation for several ablation blocks.
    """
    existing_cols = [c for c in cols if c in raw.columns]
    if not existing_cols:
        X[f"{prefix}_total_count"] = 0.0
        return X

    parsed_by_col = {
        col: raw[col].map(_parse_sanitized_item_set).reset_index(drop=True)
        for col in existing_cols
    }

    family_count_cols = []
    for fam, terms in families.items():
        terms_sanitized = {sanitize_feature_name(t) for t in terms}
        total = np.zeros(len(raw), dtype=float)
        for parsed in parsed_by_col.values():
            total += parsed.map(lambda items: len(items.intersection(terms_sanitized))).to_numpy(dtype=float)
        count_col = f"{prefix}_{fam}_count"
        flag_col = f"{prefix}_{fam}_flag"
        X[count_col] = total
        X[flag_col] = (total > 0).astype(float)
        family_count_cols.append(count_col)

    X[f"{prefix}_total_count"] = X[family_count_cols].sum(axis=1) if family_count_cols else 0.0
    return X

@dataclass
class CleanExtraFeatureEngineer:
    """Adds controlled, ML1-safe compression blocks selected in this notebook."""
    blocks: Tuple[str, ...] = ()
    columns_: List[str] = field(default_factory=list)

    def _add_nonpoly_features(self, X_base: pd.DataFrame, raw_df: pd.DataFrame) -> pd.DataFrame:
        X = X_base.copy().reset_index(drop=True)
        raw = raw_df.reset_index(drop=True).copy()

        # A. Baseline technology-family compression from the previous notebook.
        if "tech_families" in self.blocks:
            X = _add_family_features(
                X, raw, TECH_FAMILIES, TECH_FAMILY_COLS, prefix="clean_techfam"
            )

        # B. Side-coding and learning-source compression.
        if "side_learning_families" in self.blocks:
            X = _add_family_features(
                X, raw, SIDE_LEARNING_FAMILIES, SIDE_LEARNING_COLS, prefix="clean_sidelearn"
            )
            X["clean_side_income_or_business_flag"] = (
                _col(X, "clean_sidelearn_side_income_business_count", 0.0) > 0
            ).astype(float)
            X["clean_self_directed_learning_count"] = (
                _col(X, "clean_sidelearn_side_hobby_learning_count", 0.0)
                + _col(X, "clean_sidelearn_learned_online_count", 0.0)
                + _col(X, "clean_sidelearn_learned_books_media_count", 0.0)
            )

        # C. AI profile compression: combine AI adoption, trust/sentiment/time, and AI tool use.
        if "ai_profile" in self.blocks:
            X = _add_family_features(
                X, raw, AI_TOOL_FAMILIES, AI_TOOL_COLS, prefix="clean_aifam"
            )
            uses_ai = _col(X, "uses_ai", 0.0)
            ai_sent = _col(X, "ai_sentiment", 2.0)
            ai_trust = _col(X, "ai_trust", 2.0)
            ai_complex = _col(X, "ai_complex_rating", 2.0)
            search_time = _col(X, "daily_search_time", 0.0)
            answer_time = _col(X, "daily_answer_time", 0.0)

            threat_yes = raw.get("ai.job.threat", pd.Series("", index=X.index)).fillna("").astype(str).str.lower().eq("yes").astype(float)
            X["clean_ai_active_user_flag"] = (uses_ai >= 2).astype(float)
            X["clean_ai_positive_sentiment_flag"] = (ai_sent >= 3).astype(float)
            X["clean_ai_high_trust_flag"] = (ai_trust >= 3).astype(float)
            X["clean_ai_low_trust_or_threat_flag"] = ((ai_trust <= 1) | (threat_yes > 0)).astype(float)
            X["clean_ai_complex_good_flag"] = (ai_complex >= 3).astype(float)
            X["clean_ai_heavy_time_user_flag"] = ((search_time >= 3) | (answer_time >= 3)).astype(float)
            X["clean_ai_adoption_score"] = (
                uses_ai + ai_sent + ai_trust + ai_complex + _col(X, "clean_aifam_total_count", 0.0)
            ).astype(float)
            X["clean_ai_missing_profile_count"] = _missing_count(raw, MISSINGNESS_GROUPS["ai"])

        # D. Work-profile compression: compact flags built from employment, work mode, company, manager, seniority.
        if "work_profile" in self.blocks:
            emp = raw.get("employment.type", pd.Series("", index=X.index)).fillna("").astype(str)
            workloc = raw.get("work.location", pd.Series("", index=X.index)).fillna("").astype(str)
            is_full_time = emp.eq("Full-time").astype(float)
            is_freelance = emp.str.contains("freelance|self", case=False, na=False).astype(float)
            is_student = emp.str.contains("student", case=False, na=False).astype(float)
            is_jobseeker = emp.str.contains("job-seeking|job seeking", case=False, na=False).astype(float)
            is_parttime = emp.str.contains("part-time|part time", case=False, na=False).astype(float)
            is_remote = workloc.eq("Remote").astype(float)
            is_hybrid = workloc.str.contains("hybrid", case=False, na=False).astype(float)
            company = _col(X, "company_size", 0.0)
            manager = _col(X, "people_manager", 0.0)
            senior = _col(X, "senior_professional_10y", 0.0)
            senior20 = _col(X, "senior_professional_20y", 0.0)
            influence = _col(X, "tech_purchase_influence", 0.0)
            is_prof_dev = _col(X, "is_dev_professional", 1.0)

            X["clean_work_fulltime_remote"] = is_full_time * is_remote
            X["clean_work_fulltime_hybrid"] = is_full_time * is_hybrid
            X["clean_work_freelance_remote"] = is_freelance * is_remote
            X["clean_work_manager_large_company"] = manager * (company >= 6).astype(float)
            X["clean_work_manager_senior"] = manager * senior
            X["clean_work_manager_senior20"] = manager * senior20
            X["clean_work_senior_large_company"] = senior * (company >= 6).astype(float)
            X["clean_work_senior_remote_large_company"] = senior * is_remote * (company >= 6).astype(float)
            X["clean_work_solo_or_small_company"] = (company <= 1).astype(float)
            X["clean_work_student_jobseeker_parttime"] = (is_student + is_jobseeker + is_parttime).clip(upper=1)
            X["clean_work_nonprofessional_low_income_risk"] = ((is_prof_dev <= 0) | (is_student > 0) | (is_jobseeker > 0)).astype(float)
            X["clean_work_high_influence_large_company"] = (influence >= 2).astype(float) * (company >= 6).astype(float)

        # E. OS/tooling compression: dense summaries for OS, IDE, build, and collaboration ecosystems.
        if "os_tooling_families" in self.blocks:
            X = _add_family_features(
                X, raw, OS_TOOLING_FAMILIES, OS_TOOLING_COLS, prefix="clean_toolfam"
            )
            X["clean_toolfam_linux_or_apple_count"] = (
                _col(X, "clean_toolfam_linux_unix_ecosystem_count", 0.0)
                + _col(X, "clean_toolfam_apple_ecosystem_count", 0.0)
            )
            X["clean_toolfam_professional_tooling_score"] = (
                _col(X, "clean_toolfam_professional_ide_count", 0.0)
                + _col(X, "clean_toolfam_build_package_tools_count", 0.0)
                + _col(X, "clean_toolfam_collaboration_enterprise_count", 0.0)
            )

        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return X

    def fit(self, X_base: pd.DataFrame, raw_df: pd.DataFrame) -> "CleanExtraFeatureEngineer":
        X_final = self._add_nonpoly_features(X_base, raw_df)
        self.columns_ = list(X_final.columns)
        return self

    def transform(self, X_base: pd.DataFrame, raw_df: pd.DataFrame) -> pd.DataFrame:
        X = self._add_nonpoly_features(X_base, raw_df)
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

        # Align to training columns if fitted.
        if self.columns_:
            for col in self.columns_:
                if col not in X.columns:
                    X[col] = 0.0
            extra_cols = [c for c in X.columns if c not in self.columns_]
            if extra_cols:
                X = X.drop(columns=extra_cols)
            X = X[self.columns_]
        return X


def feature_count_breakdown(feature_names: List[str]) -> pd.DataFrame:
    """Summarize why the model has many features."""
    rows = []
    for f in feature_names:
        if f.startswith("clean_techfam"):
            group = "extra: technology families"
        elif f.startswith("clean_sidelearn") or f.startswith("clean_side") or f.startswith("clean_self_directed"):
            group = "extra: side/learning compression"
        elif f.startswith("clean_ai") or f.startswith("clean_aifam"):
            group = "extra: AI profile compression"
        elif f.startswith("clean_work"):
            group = "extra: work-profile compression"
        elif f.startswith("clean_toolfam"):
            group = "extra: OS/tooling compression"
        elif "_te_" in f:
            group = "base: target encodings"
        elif any(x in f for x in ["_salary_score_", "_high100_score_", "_high200_score_", "_low500_score_"]):
            group = "base: multi-select salary scores"
        elif "__" in f:
            group = "base: top multi-select binaries"
        elif f.endswith("_missing"):
            group = "base: missing indicators"
        else:
            group = "base: numeric / ordinal / engineered"
        rows.append({"feature": f, "group": group})
    return pd.DataFrame(rows).groupby("group").size().reset_index(name="n_features").sort_values("n_features", ascending=False)

print("Compression feature engineer ready.")


Compression feature engineer ready.


In [ ]:

# ===============================================================
# 6b. High-tail identification feature engineer
# ===============================================================
# This wrapper keeps the successful compression blocks from notebook 14 and adds
# only two high-tail identification tools:
# 1) smoothed upper-distribution target encodings, fitted only on the training rows;
# 2) logistic-regression probabilities for salary >100k and >150k, fitted only on training rows.

from sklearn.linear_model import LogisticRegression

UPPER_TE_COLS = [
    "region", "dev.role", "industry", "employment.type", "work.location", "company.size"
]

UPPER_TE_PAIRS = [
    ("region", "dev.role"),
    ("region", "employment.type"),
    ("region", "work.location"),
    ("region", "company.size"),
    ("dev.role", "company.size"),
    ("region", "professional_years_bucket"),
    ("dev.role", "professional_years_bucket"),
]

COMPRESSION_BLOCKS = {
    "tech_families", "work_profile", "side_learning_families", "ai_profile", "os_tooling_families"
}

HIGH_TAIL_BLOCKS = {
    "upper_distribution_te", "high100_prob", "high100_high150_prob"
}


def _clean_cat_for_tail(raw: pd.DataFrame, col: str) -> pd.Series:
    if col == "professional_years_bucket":
        if "coding.years.professional" in raw.columns:
            return exp_bucket_from_prof_years(raw["coding.years.professional"]).reset_index(drop=True)
        return pd.Series("__Missing__", index=raw.index)
    if col not in raw.columns:
        return pd.Series("__Missing__", index=raw.index)
    return raw[col].fillna("__Missing__").astype(str).reset_index(drop=True)


def _interaction_for_tail(raw: pd.DataFrame, col_a: str, col_b: str) -> pd.Series:
    a = _clean_cat_for_tail(raw, col_a)
    b = _clean_cat_for_tail(raw, col_b)
    return (a + "__x__" + b).astype(str)


@dataclass
class SmoothedUpperStats:
    q75_map: Dict[str, float]
    q90_map: Dict[str, float]
    count_map: Dict[str, int]
    global_q75: float
    global_q90: float
    smoothing: float = 12.0


@dataclass
class HighTailFeatureEngineer:
    """Add selected compression blocks and high-earner identification features.

    The class receives the already-created v6 base feature matrix, so it builds on v6
    rather than replacing it. Target-derived encodings and logistic probabilities are
    fitted only on the training part supplied to .fit().
    """
    blocks: Tuple[str, ...] = ()
    columns_: List[str] = field(default_factory=list)
    compression_engineer_: Optional[CleanExtraFeatureEngineer] = None
    upper_stats_: Dict[str, SmoothedUpperStats] = field(default_factory=dict)
    logit_scaler_: Optional[StandardScaler] = None
    logit_models_: Dict[str, Any] = field(default_factory=dict)
    logit_base_rates_: Dict[str, float] = field(default_factory=dict)

    def _compression_blocks(self) -> Tuple[str, ...]:
        return tuple([b for b in self.blocks if b in COMPRESSION_BLOCKS])

    def _fit_one_upper_stats(self, values: pd.Series, y_raw: pd.Series, smoothing: float = 12.0) -> SmoothedUpperStats:
        z = np.log(np.clip(pd.Series(y_raw).astype(float), LOW_SALARY_FLOOR, None))
        global_q75 = float(z.quantile(0.75))
        global_q90 = float(z.quantile(0.90))
        tmp = pd.DataFrame({"key": values.astype(str).fillna("__Missing__"), "z": z})
        grouped = tmp.groupby("key")["z"]
        q75_raw = grouped.quantile(0.75)
        q90_raw = grouped.quantile(0.90)
        counts = grouped.size()
        q75_map, q90_map, count_map = {}, {}, {}
        for key, n in counts.items():
            weight = float(n / (n + smoothing))
            q75_map[key] = float(weight * q75_raw.loc[key] + (1.0 - weight) * global_q75)
            q90_map[key] = float(weight * q90_raw.loc[key] + (1.0 - weight) * global_q90)
            count_map[key] = int(n)
        return SmoothedUpperStats(q75_map, q90_map, count_map, global_q75, global_q90, smoothing)

    def _fit_upper_distribution_stats(self, raw_df: pd.DataFrame, y_raw: pd.Series) -> None:
        raw = raw_df.reset_index(drop=True).copy()
        self.upper_stats_ = {}
        for col in UPPER_TE_COLS:
            values = _clean_cat_for_tail(raw, col)
            key = f"upper__{col}"
            self.upper_stats_[key] = self._fit_one_upper_stats(values, y_raw)
        for col_a, col_b in UPPER_TE_PAIRS:
            values = _interaction_for_tail(raw, col_a, col_b)
            key = f"upper__{col_a}__x__{col_b}"
            self.upper_stats_[key] = self._fit_one_upper_stats(values, y_raw)

    def _add_upper_distribution_features(self, X: pd.DataFrame, raw_df: pd.DataFrame) -> pd.DataFrame:
        if "upper_distribution_te" not in self.blocks:
            return X
        raw = raw_df.reset_index(drop=True).copy()
        for key, stats in self.upper_stats_.items():
            tail_key = key.replace("upper__", "")
            if "__x__" in tail_key:
                col_a, col_b = tail_key.split("__x__", 1)
                values = _interaction_for_tail(raw, col_a, col_b)
            else:
                values = _clean_cat_for_tail(raw, tail_key)
            safe = sanitize_feature_name(tail_key)
            vals = values.astype(str)
            X[f"tailte_{safe}_log_q75"] = vals.map(stats.q75_map).fillna(stats.global_q75).astype(float)
            X[f"tailte_{safe}_log_q90"] = vals.map(stats.q90_map).fillna(stats.global_q90).astype(float)
            X[f"tailte_{safe}_log_count"] = np.log1p(vals.map(stats.count_map).fillna(0).astype(float))
        return X

    def _fit_logistic_prob_models(self, X_for_logit: pd.DataFrame, y_raw: pd.Series) -> None:
        thresholds = []
        if "high100_prob" in self.blocks or "high100_high150_prob" in self.blocks:
            thresholds.append(("high100", HIGH_SALARY_100K))
        if "high100_high150_prob" in self.blocks:
            thresholds.append(("high150", 150_000.0))
        self.logit_models_ = {}
        self.logit_base_rates_ = {}
        if not thresholds:
            return
        self.logit_scaler_ = StandardScaler()
        X_scaled = self.logit_scaler_.fit_transform(X_for_logit)
        y = pd.Series(y_raw).astype(float)
        for label, threshold in thresholds:
            yy = (y > threshold).astype(int).to_numpy()
            rate = float(np.mean(yy))
            self.logit_base_rates_[label] = rate
            # Need both classes and enough positives/negatives. If not, use base rate.
            if yy.sum() < 5 or (len(yy) - yy.sum()) < 5:
                self.logit_models_[label] = None
                continue
            model = LogisticRegression(
                C=0.5,
                solver="liblinear",
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            )
            model.fit(X_scaled, yy)
            self.logit_models_[label] = model

    def _add_logistic_prob_features(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self.logit_models_:
            return X
        X_scaled = self.logit_scaler_.transform(X)
        for label, model in self.logit_models_.items():
            if model is None:
                prob = np.full(len(X), self.logit_base_rates_.get(label, 0.0), dtype=float)
            else:
                prob = model.predict_proba(X_scaled)[:, 1]
            X[f"tailprob_{label}"] = prob.astype(float)
        if "tailprob_high100" in X.columns and "tailprob_high150" in X.columns:
            X["tailprob_high100_x_high150"] = X["tailprob_high100"] * X["tailprob_high150"]
        return X

    def _base_plus_upper(self, X_base: pd.DataFrame, raw_df: pd.DataFrame) -> pd.DataFrame:
        X = self.compression_engineer_.transform(X_base, raw_df)
        X = self._add_upper_distribution_features(X, raw_df)
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return X

    def fit(self, X_base: pd.DataFrame, raw_df: pd.DataFrame, y_raw: pd.Series) -> "HighTailFeatureEngineer":
        self.compression_engineer_ = CleanExtraFeatureEngineer(blocks=self._compression_blocks())
        self.compression_engineer_.fit(X_base, raw_df)
        if "upper_distribution_te" in self.blocks:
            self._fit_upper_distribution_stats(raw_df, y_raw)
        X_mid = self._base_plus_upper(X_base, raw_df)
        self._fit_logistic_prob_models(X_mid, y_raw)
        X_final = self._add_logistic_prob_features(X_mid.copy())
        self.columns_ = list(X_final.columns)
        return self

    def transform(self, X_base: pd.DataFrame, raw_df: pd.DataFrame) -> pd.DataFrame:
        X_mid = self._base_plus_upper(X_base, raw_df)
        X = self._add_logistic_prob_features(X_mid.copy())
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        for col in self.columns_:
            if col not in X.columns:
                X[col] = 0.0
        extra_cols = [c for c in X.columns if c not in self.columns_]
        if extra_cols:
            X = X.drop(columns=extra_cols)
        return X[self.columns_].astype(float)


print("High-tail feature engineer ready.")


High-tail feature engineer ready.


In [ ]:
# ===============================================================
# 6. Metrics, train-only calibration, and sample weights
# ===============================================================
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def salary_bucket_summary(y_true: pd.Series, y_pred: np.ndarray) -> pd.DataFrame:
    df = pd.DataFrame({"actual": np.asarray(y_true, dtype=float), "pred": np.asarray(y_pred, dtype=float)})
    df["error"] = df["pred"] - df["actual"]
    df["sq_error"] = df["error"] ** 2
    df["abs_error"] = df["error"].abs()
    df["bucket"] = pd.cut(
        df["actual"],
        bins=[0, 500, 10_000, 30_000, 60_000, 100_000, 200_000, np.inf],
        labels=["<500", "500-10k", "10k-30k", "30k-60k", "60k-100k", "100k-200k", "200k+"],
        include_lowest=True,
    )
    out = df.groupby("bucket", observed=False).agg(
        n=("actual", "size"),
        mean_actual=("actual", "mean"),
        mean_pred=("pred", "mean"),
        rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
        mae=("abs_error", "mean"),
        mse_sum=("sq_error", "sum"),
    ).reset_index()
    out["mse_contribution_pct"] = 100 * out["mse_sum"] / df["sq_error"].sum()
    return out


# -------------------------------
# Sample-weight schemes
# -------------------------------
WEIGHT_SCHEMES = {
    "none": {"lt10k": 1.0, "gt100k": 1.0, "gt200k": 1.0, "gt500k": 1.0},
    "high_mild": {"lt10k": 1.0, "gt100k": 2.0, "gt200k": 4.0, "gt500k": 6.0},
    "high_medium": {"lt10k": 1.0, "gt100k": 3.0, "gt200k": 7.0, "gt500k": 12.0},
    "tail_balanced": {"lt10k": 2.0, "gt100k": 3.0, "gt200k": 7.0, "gt500k": 12.0},
    "tail_strong": {"lt10k": 2.5, "gt100k": 4.0, "gt200k": 10.0, "gt500k": 18.0},
}


def make_sample_weights(y_raw: pd.Series, scheme: str) -> np.ndarray:
    y = pd.Series(y_raw).astype(float).to_numpy()
    cfg = WEIGHT_SCHEMES.get(scheme)
    if cfg is None:
        raise ValueError(f"Unknown weight scheme: {scheme}")
    w = np.ones(len(y), dtype=float)
    w[y < LOW_SALARY_10K] = cfg["lt10k"]
    w[y > HIGH_SALARY_100K] = cfg["gt100k"]
    w[y > HIGH_SALARY_200K] = cfg["gt200k"]
    w[y > HIGH_SALARY_500K] = cfg["gt500k"]
    # Normalize mean weight to 1 so C remains comparable across schemes.
    w = w / np.mean(w)
    return w


def sample_weight_summary(y_raw: pd.Series, scheme: str) -> Dict[str, float]:
    w = make_sample_weights(y_raw, scheme)
    return {
        "weight_scheme": scheme,
        "weight_mean": float(np.mean(w)),
        "weight_min": float(np.min(w)),
        "weight_max": float(np.max(w)),
        "n_weight_gt1": int((w > 1.01).sum()),
    }


# -------------------------------
# Train-only calibration
# -------------------------------
def fit_train_global_ratio(pred_train_usd: np.ndarray, y_train_raw: pd.Series, cap_low: float = 0.65, cap_high: float = 2.25) -> float:
    pred = np.clip(np.asarray(pred_train_usd, dtype=float), PRED_MIN_USD, PRED_MAX_USD)
    true = np.clip(pd.Series(y_train_raw).astype(float).to_numpy(), PRED_MIN_USD, PRED_MAX_USD)
    ratios = np.clip(true / pred, cap_low, cap_high)
    return float(np.mean(ratios))


@dataclass
class BinnedRatioCalibrator:
    n_bins: int = 6
    min_bin_size: int = 25
    smoothing: float = 40.0
    cap_low: float = 0.60
    cap_high: float = 2.75
    global_factor_: float = 1.0
    bin_edges_: Optional[np.ndarray] = None
    bin_factors_: Optional[np.ndarray] = None

    def fit(self, pred_train_usd: np.ndarray, y_train_raw: pd.Series) -> "BinnedRatioCalibrator":
        pred = np.clip(np.asarray(pred_train_usd, dtype=float), PRED_MIN_USD, PRED_MAX_USD)
        true = np.clip(pd.Series(y_train_raw).astype(float).to_numpy(), PRED_MIN_USD, PRED_MAX_USD)
        pred_log = np.log(pred)
        ratios = np.clip(true / pred, self.cap_low, self.cap_high)
        self.global_factor_ = float(np.mean(ratios))
        edges = np.unique(np.quantile(pred_log, np.linspace(0, 1, self.n_bins + 1)))
        if len(edges) <= 2:
            self.bin_edges_ = np.array([-np.inf, np.inf])
            self.bin_factors_ = np.array([self.global_factor_])
            return self
        edges[0] = -np.inf
        edges[-1] = np.inf
        self.bin_edges_ = edges
        bin_ids = np.digitize(pred_log, edges[1:-1], right=True)
        factors = []
        for b in range(len(edges) - 1):
            mask = bin_ids == b
            n = int(mask.sum())
            if n < self.min_bin_size:
                factors.append(self.global_factor_)
            else:
                raw_factor = float(np.mean(ratios[mask]))
                smoothed = (raw_factor * n + self.global_factor_ * self.smoothing) / (n + self.smoothing)
                factors.append(float(np.clip(smoothed, self.cap_low, self.cap_high)))
        self.bin_factors_ = np.array(factors)
        return self

    def apply(self, pred_usd: np.ndarray) -> np.ndarray:
        pred = np.clip(np.asarray(pred_usd, dtype=float), PRED_MIN_USD, PRED_MAX_USD)
        if self.bin_edges_ is None or self.bin_factors_ is None:
            return np.clip(pred * self.global_factor_, PRED_MIN_USD, PRED_MAX_USD)
        pred_log = np.log(pred)
        bin_ids = np.digitize(pred_log, self.bin_edges_[1:-1], right=True)
        factors = self.bin_factors_[bin_ids]
        return np.clip(pred * factors, PRED_MIN_USD, PRED_MAX_USD)


def evaluate_calibrations(
    target_transformer: TargetTransformer,
    pred_train_scaled: np.ndarray,
    pred_val_scaled: np.ndarray,
    y_train_raw_used: pd.Series,
    y_val_raw_eval: pd.Series,
) -> Dict[str, Any]:
    pred_train_usd = target_transformer.inverse_transform_to_usd(pred_train_scaled)
    pred_val_usd_none = target_transformer.inverse_transform_to_usd(pred_val_scaled)

    records = []
    records.append({
        "calibration": "none",
        "factor_or_info": "direct",
        "pred_val_usd": pred_val_usd_none,
        "val_rmse": rmse(y_val_raw_eval, pred_val_usd_none),
    })

    # Global train-only ratio calibration. Unlike validation_multiplier, this is fitted only on training residuals.
    global_ratio = fit_train_global_ratio(pred_train_usd, y_train_raw_used)
    pred_global = np.clip(pred_val_usd_none * global_ratio, PRED_MIN_USD, PRED_MAX_USD)
    records.append({
        "calibration": "train_global_ratio",
        "factor_or_info": global_ratio,
        "pred_val_usd": pred_global,
        "val_rmse": rmse(y_val_raw_eval, pred_global),
    })

    # Binned train-only ratio calibration, designed to correct underprediction differently across prediction ranges.
    binned = BinnedRatioCalibrator(n_bins=6).fit(pred_train_usd, y_train_raw_used)
    pred_binned = binned.apply(pred_val_usd_none)
    records.append({
        "calibration": "train_binned_ratio",
        "factor_or_info": {
            "global_factor": binned.global_factor_,
            "bin_factors": binned.bin_factors_.tolist() if binned.bin_factors_ is not None else None,
        },
        "pred_val_usd": pred_binned,
        "val_rmse": rmse(y_val_raw_eval, pred_binned),
        "calibrator": binned,
    })

    best = min(records, key=lambda r: r["val_rmse"])
    return {"records": records, "best": best}

print("Metrics, sample weights, and train-only calibration helpers ready.")


Metrics, sample weights, and train-only calibration helpers ready.


In [ ]:

# ===============================================================
# 7. Modeling helpers — high-tail identification ablation
# ===============================================================
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, KFold

BASE_FEATURE_PROFILE = "top5_dense"
BASE_TARGET_VARIANT = "log_floor500"
BASE_WEIGHT_SCHEME = "high_mild"
BASE_SVR_PARAMS = {"C": 0.2, "gamma": 0.0007, "epsilon": 0.05}

BASE_FEATURE_CACHE = {}


def prepare_features(
    train_df_fit: pd.DataFrame,
    val_df_eval: pd.DataFrame,
    kaggle_df_optional: Optional[pd.DataFrame],
    feature_blocks: Tuple[str, ...],
    feature_profile: str = BASE_FEATURE_PROFILE,
    target_variant: str = BASE_TARGET_VARIANT,
):
    """Prepare feature matrices.

    The v6 base feature extractor is fitted only once per training frame and reused
    across validation/internal/Kaggle transformations. Extra compression blocks are
    still fitted on the training rows only.
    """
    y_train_fit_raw = train_df_fit[TARGET].astype(float).reset_index(drop=True)
    base_cache_key = (id(train_df_fit), feature_profile)

    if base_cache_key not in BASE_FEATURE_CACHE:
        base_pre = TailFocusedSVRPreprocessor(feature_profile=feature_profile)
        base_pre.fit(train_df_fit.drop(columns=[TARGET]), y_train_fit_raw)
        X_train_base = base_pre.transform(train_df_fit.drop(columns=[TARGET]))
        BASE_FEATURE_CACHE[base_cache_key] = {
            "base_preprocessor": base_pre,
            "X_train_base": X_train_base,
        }

    cached = BASE_FEATURE_CACHE[base_cache_key]
    base_pre = cached["base_preprocessor"]
    X_train_base = cached["X_train_base"].copy()
    X_val_base = base_pre.transform(val_df_eval.drop(columns=[TARGET]))
    X_kaggle_base = base_pre.transform(kaggle_df_optional) if kaggle_df_optional is not None else None

    extra = HighTailFeatureEngineer(blocks=feature_blocks)
    extra.fit(X_train_base, train_df_fit.drop(columns=[TARGET]), y_train_fit_raw)
    X_train = extra.transform(X_train_base, train_df_fit.drop(columns=[TARGET]))
    X_val = extra.transform(X_val_base, val_df_eval.drop(columns=[TARGET]))
    X_kaggle = extra.transform(X_kaggle_base, kaggle_df_optional) if X_kaggle_base is not None else None

    scaler_x = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler_x.fit_transform(X_train), columns=X_train.columns)
    X_val_scaled = pd.DataFrame(scaler_x.transform(X_val), columns=X_train.columns)
    X_kaggle_scaled = pd.DataFrame(scaler_x.transform(X_kaggle), columns=X_train.columns) if X_kaggle is not None else None

    target_tf = TargetTransformer(target_variant).fit(y_train_fit_raw)
    y_train_scaled = pd.Series(target_tf.transform(y_train_fit_raw))

    return {
        "base_preprocessor": base_pre,
        "extra_engineer": extra,
        "scaler_x": scaler_x,
        "target_tf": target_tf,
        "X_train": X_train_scaled,
        "X_val": X_val_scaled,
        "X_kaggle": X_kaggle_scaled,
        "y_train_scaled": y_train_scaled,
        "y_train_raw_used": y_train_fit_raw,
        "feature_names": list(X_train.columns),
        "target_variant": target_variant,
    }

def apply_lasso_selection(X_train, y_train_scaled, X_val, X_kaggle=None, top_n: Optional[int] = None):
    """Optional course-safe feature selection: LASSO coefficients select the strongest features."""
    if top_n is None or top_n >= X_train.shape[1]:
        return X_train, X_val, X_kaggle, list(X_train.columns), None

    alpha_grid = {"alpha": np.logspace(-4, -1, 7)}
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(
        Lasso(max_iter=20000, random_state=RANDOM_STATE),
        param_grid=alpha_grid,
        scoring="neg_mean_squared_error",
        cv=cv,
        n_jobs=-1,
    )
    search.fit(X_train, y_train_scaled)
    coef = pd.Series(np.abs(search.best_estimator_.coef_), index=X_train.columns)
    selected_cols = coef.sort_values(ascending=False).head(top_n).index.tolist()

    return (
        X_train[selected_cols],
        X_val[selected_cols],
        X_kaggle[selected_cols] if X_kaggle is not None else None,
        selected_cols,
        search.best_params_,
    )


def fit_predict_svr(prepared, lasso_top_n: Optional[int] = None):
    X_train, X_val, X_kaggle, selected_cols, lasso_params = apply_lasso_selection(
        prepared["X_train"],
        prepared["y_train_scaled"],
        prepared["X_val"],
        prepared["X_kaggle"],
        top_n=lasso_top_n,
    )
    model = SVR(kernel="rbf", cache_size=200, max_iter=10000, **BASE_SVR_PARAMS)
    sample_weight = make_sample_weights(prepared["y_train_raw_used"], BASE_WEIGHT_SCHEME)
    model.fit(X_train, prepared["y_train_scaled"], sample_weight=sample_weight)

    pred_val_scaled = model.predict(X_val)
    pred_val = prepared["target_tf"].inverse_transform_to_usd(pred_val_scaled)

    pred_kaggle = None
    if X_kaggle is not None:
        pred_kaggle_scaled = model.predict(X_kaggle)
        pred_kaggle = prepared["target_tf"].inverse_transform_to_usd(pred_kaggle_scaled)

    return model, pred_val, pred_kaggle, selected_cols, lasso_params


def prediction_distribution(pred: np.ndarray) -> Dict[str, float]:
    pred = np.asarray(pred, dtype=float)
    return {
        "pred_mean": float(np.mean(pred)),
        "pred_median": float(np.median(pred)),
        "pred_p95": float(np.quantile(pred, 0.95)),
        "pred_p99": float(np.quantile(pred, 0.99)),
        "pred_max": float(np.max(pred)),
        "n_pred_gt100k": int((pred > 100_000).sum()),
        "n_pred_gt150k": int((pred > 150_000).sum()),
        "n_pred_gt200k": int((pred > 200_000).sum()),
    }


def evaluate_feature_block_candidate(name: str, feature_blocks: Tuple[str, ...], lasso_top_n: Optional[int] = None, target_variant: str = BASE_TARGET_VARIANT):
    start = time.time()
    prepared = prepare_features(
        train_df_fit=df_train,
        val_df_eval=df_val,
        kaggle_df_optional=df_kaggle,
        feature_blocks=feature_blocks,
        target_variant=target_variant,
    )
    model, pred_val, pred_kaggle, selected_cols, lasso_params = fit_predict_svr(prepared, lasso_top_n=lasso_top_n)

    record = {
        "candidate": name,
        "feature_blocks": ",".join(feature_blocks) if feature_blocks else "base_v6",
        "target_variant": target_variant,
        "lasso_top_n": lasso_top_n if lasso_top_n is not None else "none",
        "lasso_best_params": json.dumps(lasso_params) if lasso_params is not None else "",
        "n_features_before_selection": prepared["X_train"].shape[1],
        "n_features_used": len(selected_cols),
        "val_rmse": rmse(y_val_raw, pred_val),
        "val_mae": float(mean_absolute_error(y_val_raw, pred_val)),
        "fit_seconds": float(time.time() - start),
        **prediction_distribution(pred_val),
    }

    if pred_kaggle is not None:
        kg = prediction_distribution(pred_kaggle)
        record.update({f"kaggle_{k}": v for k, v in kg.items()})

    return record, pred_val, pred_kaggle, prepared, model

print("Modeling helpers ready.")


Modeling helpers ready.


In [23]:

# ===============================================================
# 8. Main high-tail identification ablation
# ===============================================================
# The current best clean foundation is tech + work compression. We now test the
# faster high-earner identification additions first. Logistic probability features
# are implemented but optional because they can make RBF-SVR much slower.

BASE_BLOCKS = ("tech_families", "work_profile")
UPPER_BLOCKS = BASE_BLOCKS + ("upper_distribution_te",)

CANDIDATES = [
    ("BASE_V6_REPRODUCTION", tuple(), None, "log_floor500"),
    ("BASE_TECH_WORK", BASE_BLOCKS, None, "log_floor500"),
    ("TECH_WORK_PLUS_UPPER_Q75_Q90", UPPER_BLOCKS, None, "log_floor500"),
]

if RUN_SQRT_TARGET_CANDIDATE:
    CANDIDATES.append(("TECH_WORK_PLUS_UPPER_Q75_Q90_SQRT_TARGET", UPPER_BLOCKS, None, "sqrt_floor500"))

if RUN_LOGISTIC_PROBABILITY_FEATURES:
    CANDIDATES.extend([
        ("TECH_WORK_PLUS_HIGH100_PROB", BASE_BLOCKS + ("high100_prob",), None, "log_floor500"),
        ("TECH_WORK_PLUS_HIGH100_HIGH150_PROB", BASE_BLOCKS + ("high100_high150_prob",), None, "log_floor500"),
        ("TECH_WORK_PLUS_UPPER_AND_HIGH100", UPPER_BLOCKS + ("high100_prob",), None, "log_floor500"),
        ("TECH_WORK_PLUS_UPPER_AND_HIGH100_SQRT_TARGET", UPPER_BLOCKS + ("high100_prob",), None, "sqrt_floor500"),
    ])

all_records = []
candidate_lookup = {
    name: {"blocks": blocks, "lasso_top_n": lasso_top_n, "target_variant": target_variant}
    for name, blocks, lasso_top_n, target_variant in CANDIDATES
}
feature_names_by_candidate = {}

for name, blocks, lasso_top_n, target_variant in CANDIDATES:
    print(f"Running {name} ...")
    rec, pred_val, pred_kaggle, prepared, model = evaluate_feature_block_candidate(
        name, blocks, lasso_top_n=lasso_top_n, target_variant=target_variant
    )
    all_records.append(rec)
    feature_names_by_candidate[name] = prepared["feature_names"]
    print(f"  validation RMSE = {rec['val_rmse']:.2f}; features used = {rec['n_features_used']}; target = {target_variant}")

results = pd.DataFrame(all_records).sort_values("val_rmse").reset_index(drop=True)
display(results)
results.to_csv(OUTPUT_DIR / "high_tail_identification_ablation_results.csv", index=False)

selected_row = results.iloc[0]
selected_name = selected_row["candidate"]
selected_info = candidate_lookup[selected_name]
selected_blocks = selected_info["blocks"]
selected_lasso_top_n = selected_info["lasso_top_n"]
selected_target_variant = selected_info["target_variant"]

print("Selected candidate:", selected_name)
print("Selected blocks:", selected_blocks)
print("Selected target variant:", selected_target_variant)
print("Selected validation RMSE:", selected_row["val_rmse"])

breakdown = feature_count_breakdown(feature_names_by_candidate[selected_name])
display(breakdown)
breakdown.to_csv(OUTPUT_DIR / "selected_feature_count_breakdown.csv", index=False)


Running BASE_V6_REPRODUCTION ...
  validation RMSE = 36900.79; features used = 376; target = log_floor500
Running BASE_TECH_WORK ...
  validation RMSE = 36688.26; features used = 403; target = log_floor500
Running TECH_WORK_PLUS_UPPER_Q75_Q90 ...
  validation RMSE = 36617.48; features used = 442; target = log_floor500


,candidate,feature_blocks,target_variant,lasso_top_n,lasso_best_params,n_features_before_selection,n_features_used,val_rmse,val_mae,fit_seconds,...,n_pred_gt150k,n_pred_gt200k,kaggle_pred_mean,kaggle_pred_median,kaggle_pred_p95,kaggle_pred_p99,kaggle_pred_max,kaggle_n_pred_gt100k,kaggle_n_pred_gt150k,kaggle_n_pred_gt200k
0,TECH_WORK_PLUS_UPPER_Q75_Q90,"tech_families,work_profile,upper_distribution_te",log_floor500,none,,442,442,36617.480249,22633.022267,8.501196,...,0,0,46914.250673,42671.443236,96502.128221,123234.816384,172696.659713,28,1,0
1,BASE_TECH_WORK,"tech_families,work_profile",log_floor500,none,,403,403,36688.255657,22915.925749,7.578027,...,1,0,47007.881334,41985.350007,98138.943294,123456.763282,181141.294145,28,1,0
2,BASE_V6_REPRODUCTION,base_v6,log_floor500,none,,376,376,36900.793838,23053.568287,12.467272,...,1,0,46887.964279,41238.230142,96346.357837,122974.026278,172484.880713,27,1,0


Selected candidate: TECH_WORK_PLUS_UPPER_Q75_Q90
Selected blocks: ('tech_families', 'work_profile', 'upper_distribution_te')
Selected target variant: log_floor500
Selected validation RMSE: 36617.480249218315


,group,n_features
2,base: numeric / ordinal / engineered,117
1,base: multi-select salary scores,105
3,base: target encodings,100
4,base: top multi-select binaries,75
0,base: missing indicators,18
5,extra: technology families,15
6,extra: work-profile compression,12


In [24]:
if RUN_INTERNAL_FINAL:

    # ===============================================================
    # 9. Internal-test and bucket diagnostics for the selected candidate
    # ===============================================================
    # selected_blocks and selected_lasso_top_n were chosen in the ablation cell above.
    print("Selected candidate:", selected_name)
    print("Selected blocks:", selected_blocks)

    prepared_internal = prepare_features(
        train_df_fit=df_train,
        val_df_eval=df_internal_test,
        kaggle_df_optional=df_kaggle,
        feature_blocks=selected_blocks,
        target_variant=selected_target_variant,
    )

    model_internal, pred_internal, pred_kaggle_selected, selected_cols_internal, lasso_params_internal = fit_predict_svr(
        prepared_internal,
        lasso_top_n=selected_lasso_top_n
    )

    internal_rmse = rmse(y_internal_test_raw, pred_internal)
    internal_mae = mean_absolute_error(y_internal_test_raw, pred_internal)

    print("Selected candidate:", selected_name)
    print("Internal-test RMSE:", round(internal_rmse, 2))
    print("Internal-test MAE:", round(internal_mae, 2))
    print("Selected feature count:", len(selected_cols_internal))

    bucket_internal = salary_bucket_summary(y_internal_test_raw, pred_internal)
    display(bucket_internal)
    bucket_internal.to_csv(OUTPUT_DIR / "high_tail_selected_internal_bucket_diagnostics.csv", index=False)

    # Compare prediction distribution to current best Kaggle submission, if available.
    best_submission_path = Path("submission_weighted_tail_selected_refit_train_val.csv")
    if best_submission_path.exists():
        current_best_submission = pd.read_csv(best_submission_path)
        print("\nCurrent best Kaggle prediction distribution:")
        display(current_best_submission[TARGET].describe().to_frame().T)
        print("\nSelected model Kaggle prediction distribution:")
        display(pd.Series(pred_kaggle_selected, name=TARGET).describe().to_frame().T)

        comparison_dist = pd.DataFrame([
            {"source": "current_best_submission", **prediction_distribution(current_best_submission[TARGET].to_numpy())},
            {"source": "selected_clean_model", **prediction_distribution(pred_kaggle_selected)},
        ])
        display(comparison_dist)
        comparison_dist.to_csv(OUTPUT_DIR / "compression_kaggle_prediction_distribution_comparison.csv", index=False)
else:
    print("RUN_INTERNAL_FINAL=False, so internal-test diagnostics are skipped in the default quick run.")


Selected candidate: TECH_WORK_PLUS_UPPER_Q75_Q90
Selected blocks: ('tech_families', 'work_profile', 'upper_distribution_te')
Selected candidate: TECH_WORK_PLUS_UPPER_Q75_Q90
Internal-test RMSE: 50594.4
Internal-test MAE: 23197.11
Selected feature count: 442


,bucket,n,mean_actual,mean_pred,rmse,mae,mse_sum,mse_contribution_pct
0,<500,12,249.250000,30782.303948,35729.157702,30533.053948,1.531887e+10,1.587378
1,500-10k,58,2718.844828,39716.491566,42767.004580,36997.646738,1.060830e+11,10.992573
2,10k-30k,73,21180.684932,28266.415918,15232.837496,10780.439269,1.693887e+10,1.755247
3,30k-60k,137,45077.379562,48845.272817,20354.975271,15385.979866,5.676253e+10,5.881870
4,60k-100k,74,74173.378378,62842.574799,22650.274370,18368.272315,3.796458e+10,3.933982
5,100k-200k,21,132373.000000,85571.658445,57123.592974,52187.625194,6.852520e+10,7.100747
6,200k+,2,557293.000000,115789.317663,575955.381296,441503.682337,6.634492e+11,68.748204


In [25]:
if RUN_INTERNAL_FINAL:

    # ===============================================================
    # 10. Small ML1 model comparison on the selected feature set
    # ===============================================================
    # This keeps the project documentation complete without turning the notebook into a huge search.
    # All models are from the ML1 course family: Ridge, LASSO, Elastic Net, KNN, SVR.

    prepared = prepare_features(
        train_df_fit=df_train,
        val_df_eval=df_val,
        kaggle_df_optional=None,
        feature_blocks=selected_blocks,
        target_variant=selected_target_variant,
    )
    X_train = prepared["X_train"]
    X_val = prepared["X_val"]
    y_train_scaled = prepared["y_train_scaled"]
    target_tf = prepared["target_tf"]
    weights = make_sample_weights(prepared["y_train_raw_used"], BASE_WEIGHT_SCHEME)

    models = {
        "Ridge": Ridge(alpha=10.0),
        "LASSO": Lasso(alpha=0.001, max_iter=20000, random_state=RANDOM_STATE),
        "ElasticNet": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=20000, random_state=RANDOM_STATE),
        "KNN": KNeighborsRegressor(n_neighbors=15, weights="distance"),
        "SVR_RBF_v6_params": SVR(kernel="rbf", cache_size=200, max_iter=10000, **BASE_SVR_PARAMS),
    }

    model_records = []
    for model_name, model in models.items():
        print("Fitting", model_name)
        if model_name in ["KNN"]:
            model.fit(X_train, y_train_scaled)
        else:
            try:
                model.fit(X_train, y_train_scaled, sample_weight=weights)
            except TypeError:
                model.fit(X_train, y_train_scaled)

        pred_val_scaled = model.predict(X_val)
        pred_val_usd = target_tf.inverse_transform_to_usd(pred_val_scaled)

        model_records.append({
            "model": model_name,
            "val_rmse": rmse(y_val_raw, pred_val_usd),
            "val_mae": float(mean_absolute_error(y_val_raw, pred_val_usd)),
            **prediction_distribution(pred_val_usd),
        })

    model_compare_df = pd.DataFrame(model_records).sort_values("val_rmse").reset_index(drop=True)
    display(model_compare_df)
    model_compare_df.to_csv(OUTPUT_DIR / "compression_ml1_model_comparison.csv", index=False)
else:
    print("RUN_INTERNAL_FINAL=False, so the small ML1 model comparison is skipped in the default quick run.")


Fitting Ridge
Fitting LASSO
Fitting ElasticNet
Fitting KNN
Fitting SVR_RBF_v6_params


,model,val_rmse,val_mae,pred_mean,pred_median,pred_p95,pred_p99,pred_max,n_pred_gt100k,n_pred_gt150k,n_pred_gt200k
0,SVR_RBF_v6_params,36617.480249,22633.022267,47119.772002,41905.909439,95503.450183,126419.043991,149282.375538,15,0,0
1,KNN,42407.005715,26572.605393,31700.341707,28671.132465,60038.340537,81393.194297,109436.055977,1,0,0
2,LASSO,54329.552311,33110.032250,46270.542404,28441.919339,144355.905890,283636.407405,355665.514955,46,17,8
3,ElasticNet,56198.080025,34017.759530,47122.656626,28777.236476,151640.284985,298833.361266,395422.242799,45,20,8
4,Ridge,58218.805887,34821.149583,47851.179984,29420.181724,145394.271879,328517.773979,442480.324603,47,18,10


In [29]:
if RUN_INTERNAL_FINAL:

    # ===============================================================
    # 11. Refit selected model on train + validation and create submission
    # ===============================================================
    # Validation and internal-test targets stayed raw throughout evaluation.
    # The final model is refit on train + validation only after candidate choice.

    df_trainval_refit = pd.concat([df_train, df_val], axis=0).reset_index(drop=True)

    prepared_final = prepare_features(
        train_df_fit=df_trainval_refit,
        val_df_eval=df_internal_test,      # placeholder for internal diagnostics
        kaggle_df_optional=df_kaggle,
        feature_blocks=selected_blocks,
        target_variant=selected_target_variant,
    )

    final_model, pred_internal_refit, pred_kaggle_final, selected_cols_final, lasso_params_final = fit_predict_svr(
        prepared_final,
        lasso_top_n=selected_lasso_top_n
    )

    submission = pd.DataFrame({
        ID_COL: df_kaggle[ID_COL].values,
        TARGET: pred_kaggle_final,
    })

    submission_path = OUTPUT_DIR / "submission_clean_v6_high_tail_identification.csv"
    submission.to_csv(submission_path, index=False)

    print("Saved submission:", submission_path.resolve())
    print("Final Kaggle prediction distribution:")
    display(submission[TARGET].describe().to_frame().T)

    summary = {
        "selected_candidate": selected_name,
        "selected_blocks": list(selected_blocks),
        "selected_target_variant": selected_target_variant,
        #"selected_feature_count_breakdown": selected_feature_breakdown.to_dict(orient="records"),
        "selected_lasso_top_n": selected_lasso_top_n,
        #"base_validation_rmse": base_rmse,
        # "selected_validation_rmse": selected_rmse,
        # "selected_internal_test_rmse_before_refit": internal_rmse,
        # "n_features_final": len(selected_cols_final),
        # "submission_path": str(submission_path),
        # "important_note": "Submit this only if diagnostics are at least as good as the current v6 benchmark; otherwise keep the current best public submission.",
    }
    with open(OUTPUT_DIR / "high_tail_selected_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
else:
    print("RUN_INTERNAL_FINAL=False, so final refit and submission creation are skipped in the default quick run.")
    print("After selecting a candidate, set RUN_INTERNAL_FINAL=True and rerun from a fresh kernel to create the final submission.")


Saved submission: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\clean_v6_high_tail_outputs\submission_clean_v6_high_tail_identification.csv
Final Kaggle prediction distribution:


,count,mean,std,min,25%,50%,75%,max
annual.pay.usd,628.0,46714.122791,26774.570706,5626.375704,26841.732995,42795.528493,61405.56327,171761.443835


{
  "selected_candidate": "TECH_WORK_PLUS_UPPER_Q75_Q90",
  "selected_blocks": [
    "tech_families",
    "work_profile",
    "upper_distribution_te"
  ],
  "selected_target_variant": "log_floor500",
  "selected_lasso_top_n": null
}


## How to interpret the results

Use `BASE_V6_REPRODUCTION` as the benchmark and `BASE_TECH_WORK` as the strongest clean feature foundation from the previous notebook.

This notebook tests whether high-earner identification features can improve the model without making it aggressive. The fast default run tests upper-distribution encodings. The sqrt target and logistic probability features are implemented but optional because they slowed down the RBF-SVR fit during testing:

- `upper_distribution_te`: smoothed q75/q90 log-salary encodings for high-salary categories and interactions.
- `high100_prob`: optional logistic-regression probability that the salary is above 100k.
- `high100_high150_prob`: optional probabilities for above 100k and above 150k.
- `sqrt_floor500`: a less-compressive target transformation tested only for the strongest feature set.

The default run saves:

- `clean_v6_high_tail_outputs/high_tail_identification_ablation_results.csv`
- `clean_v6_high_tail_outputs/selected_feature_count_breakdown.csv`

If the selected candidate looks promising, set:

`RUN_INTERNAL_FINAL = True`

Then restart the kernel and rerun the notebook to create internal-test diagnostics and:

`clean_v6_high_tail_outputs/submission_clean_v6_high_tail_identification.csv`

Submit only if the validation/internal diagnostics are at least as good as the current v6 benchmark and the high-salary bucket improves without breaking the middle of the salary distribution.
